# Question 2 — Motion or Memory?
AI651 Assignment 2 · Hasan Amjad (25280042)

The notebook calls the functions in `q2/`. Each step saves its numbers to `results/` and its plots to `figures/`, so the report can be written from those files.

In [ ]:
%load_ext autoreload
%autoreload 2
import platform, time, torch
from q2.common import *
from q2.transforms import *
from q2.models import *

DEVICE = get_device()
print("torch", torch.__version__, "| device:", DEVICE, "|", platform.platform(), platform.processor())

## Step 1 · Setup
### Table 1 — checking the transform and window functions
Uses synthetic tensors only. Every row must say OK before going further.

In [ ]:
from q2.checks import run_table1, print_table1
table1 = run_table1()
print_table1(table1)
save_json(table1, "table1.json");

### Load Moving MNIST and split by video index
First run downloads ~800 MB into `Question2/data/` (ignored by git).

In [ ]:
data = load_moving_mnist()
train_u8, val_u8, test_u8 = splits(data)
print("all", tuple(data.shape), data.dtype, "| train", len(train_u8), "| val", len(val_u8), "| test", len(test_u8))

### Figure 1 — motion statistics of the training data
Farneback flow on the first 50 training videos, 19 consecutive pairs each (950 pairs).

In [ ]:
from q2.flow import pooled_flow, figure1
dx, dy, n_pairs = pooled_flow(train_u8, n_videos=50)
fig, fig1_stats = figure1(dx, dy, FIG_DIR / "fig1_flow_stats.png")
fig1_stats["n_pairs"] = n_pairs
save_json(fig1_stats, "fig1_stats.json")
print(f"pairs = {n_pairs}, pixels kept = {fig1_stats['n_pixels']:,}, median speed = {fig1_stats['median_speed']:.3g} px/frame")
print("2x ->", round(2*fig1_stats['median_speed'], 3), "px/frame;  3x ->", round(3*fig1_stats['median_speed'], 3), "px/frame")
print("direction fractions per 20-degree bin:", fig1_stats["direction_fractions"])

### Model A — parameter check
The handout says model A has exactly 480,784 trainable parameters.

In [ ]:
mA = model_A()
print("model A params:", n_params(mA))
assert n_params(mA) == 480_784
with torch.no_grad():
    out = mA(torch.rand(2, 5, 1, 64, 64))
print("output shape:", tuple(out.shape))